# SAPA Edge — Test Notebook

Notebook ini test edge server SAPA langsung di Jupyter.

## Sebelum mulai

1. Pastikan dependencies sudah terinstall:
   ```
   pip install -r edge_server/requirements.txt
   pip install ipython pillow
   ```

2. Edit `CONFIG` di file `edge_all_in_one.py` (di sebelah notebook ini) sesuai environment Anda.

## Cell 1 — Load semua module

In [ ]:
# Import semua dari edge_all_in_one.py
%run edge_all_in_one.py

# atau kalau cell di atas tidak jalan, pakai ini:
# import sys
# sys.path.insert(0, '.')
# from edge_all_in_one import *

## Cell 2 — Cek dependency

In [ ]:
test_dependencies()

## Cell 3 — Test webcam

Capture 1 frame dari webcam dan tampilkan inline di notebook.

In [ ]:
test_camera()

## Cell 4 — Capture foto referensi karyawan (buat test_faces folder)

Capture wajah Anda 3-5 kali dengan ekspresi/sudut berbeda. Tiap orang dapat 1 file.

In [ ]:
# Capture wajah Anda dengan ID 'EMP001'
capture_face_to_file('EMP001', './test_faces')

# Boleh capture orang lain juga:
# capture_face_to_file('EMP002', './test_faces')
# capture_face_to_file('EMP003', './test_faces')

## Cell 5 — Test recognition di gambar diam

Pakai foto Anda sendiri (atau foto lain) untuk test apakah recognition jalan.
Kotak hijau = match, kotak merah = unknown.

In [ ]:
# Test pakai foto yang baru di-capture
test_static_image('./test_faces/EMP001.jpg', faces_dir='./test_faces')

## Cell 6 — LIVE realtime mode (LOKAL — tanpa VPS)

Buka window OpenCV "SAPA Edge" yang menampilkan kamera live dengan kotak hijau/merah.

Foto referensi diambil dari folder `./test_faces/`. Tekan **Q** atau **Esc** untuk berhenti.

In [ ]:
# Mode lokal: tanpa VPS, MQTT, atau report. Hanya recognition + display.
test_realtime_local(faces_dir='./test_faces', max_seconds=60)

## Cell 7 — LIVE realtime mode (DENGAN VPS)

Auto-sync foto dari VPS, report match ke `/api/edge/face-match`, publish heartbeat ke MQTT.

**SEBELUM JALANKAN:** edit `CONFIG` di `edge_all_in_one.py`:
- `SAPA_API_BASE` ke URL VPS Anda
- `EDGE_INGEST_KEY` ke kunci dari Secret VPS
- `MQTT_BROKER`, `MQTT_USERNAME`, `MQTT_PASSWORD` (atau set `ENABLE_MQTT=False` untuk skip)

In [ ]:
# Mode produksi-lite: sync dari VPS + report, jalan 2 menit
test_realtime_with_vps(max_seconds=120)

## Cell 8 — Mode produksi penuh (no timeout)

Jalan terus sampai Ctrl+C atau Q. Untuk deploy beneran biasanya dijalankan via NSSM/systemd.

In [ ]:
# run_full_service()  # uncomment kalau mau coba

## Cell 9 — Inspect cache embeddings.npz

Lihat siapa saja yang sudah ter-encode.

In [ ]:
rec = FaceRecognizer(cache_path=CONFIG['CACHE_PATH'])
rec.load_cache()
print(f'Total embeddings: {rec.known_count}')
for label in rec._labels:
    print(f'  - {label}')

## Cell 10 — Reset cache (kalau perlu rebuild dari nol)

In [ ]:
import os
if os.path.exists(CONFIG['CACHE_PATH']):
    os.remove(CONFIG['CACHE_PATH'])
    print('Cache deleted')
else:
    print('No cache to delete')